In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torchvision import datasets,transforms

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.CIFAR10(
    root = 'data',
    train=True,
    transform = transform,
    download = True
)

test_data = datasets.CIFAR10(
    root = 'data',
    train=False,
    transform = transform,
    download = True
)


train_loader = DataLoader(
    train_data,
    batch_size = 64,
    shuffle = True
)

test_loader = DataLoader(
    test_data,
    batch_size = 64,
    shuffle=False
)

In [ ]:
print(len(train_data))
print(len(test_data))

50000
10000


In [ ]:
image,label = train_data[0]

print(label)

6


In [ ]:
image.shape

torch.Size([3, 32, 32])

In [ ]:
train_data.classes

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [ ]:
class Model(nn.Module):
  def __init__(self) :
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=3
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=128*2*2,out_features=128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128,10)
    )

  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x


In [ ]:
model = Model()

criterion = nn.CrossEntropyLoss()
optmizer = optim.Adam(model.parameters(),lr=0.001)


In [ ]:
epochs = 5

for epoch in range(epochs):
  model.train()
  running_loss = 0.0

  for images,labels in train_loader:
    logits = model(images)
    loss = criterion(logits,labels)

    optmizer.zero_grad()
    loss.backward()
    optmizer.step()

    running_loss += loss.item()

average_loss = running_loss /len(train_loader)

print('epochs',epochs)
print('average loss :',average_loss)


epochs 5
average loss : 1.1054184083895915


In [ ]:
correct = 0
total = 0
model.eval()

with torch.no_grad():
  for images,labels in test_loader:
    logits = model(images)
    predictions = logits.argmax(dim=1)

    total += labels.size(0)
    correct += (predictions == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 62.88%


In [ ]:
class_names = ['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']



image, true_lable = test_data[1]

model.eval()
with torch.no_grad():
  output = model(image.unsqueeze(0))
  predicted_lable = output.argmax(dim=1).item()

print("Predicted class:", class_names[predicted_lable])
print("Actual class:", class_names[true_lable])

Predicted class: ship
Actual class: ship


In [ ]:
import matplotlib.pyplot as plt
image, label = train_data[0]

plt.imshow(img)
plt.title(class_names[label])
plt.axis("off")
plt.show()

print("Image shape:", image.shape)
print("Label:", label, "-", class_names[label])

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'